**Прогнозирование с использованием ML**

В данном блокноте мы переработаем исходную поставленную ранее задачу по прогнозированию доходности в экстремальных условиях. Мы столкнулись с проблемой, когда пытались построить портфель ценных бумаг на основе прогнозов (лонг/шорт), хотя метрики показывали неплохие результаты для градиентного бустинга. Здесь мы попробуем подойти к проблеме прогнозирования с другой стороны - нашей целью будет исследование, какие методы машинного обучения лучше работают на ряде признаков, используемых в прогнозировании. В основном это будут технические признаки. Чтобы не лезть слишком далеко, мы не будем формировать портфель, а ограничимся лишь корректными прогнозами на основе временных рядов. Также планируется применить методы снижения размерности для выделения наиболее информативных признаков. Было бы неплохо также добавить в ислледования анализ новостей и социальных сетей с помощью нейронных сетей, но это пока в перспективе.

Итак, вновь начнём с загрузки временного ряда (свечей) с сайта Московской биржи - вновь выбираем наиболее ликвидные бумаги (SBER, GAZP, LKOH, NVTK, MGNT), которые имеют высокий вес в MOEX и которые не имеют ввиду высокой ликвидности широких спредов. Кроме того, для этих эмитентов имеется широкий диапазон торговли, что очень благоприятно для анализа. Однако, 5 эмитентов - мало, сигнал будет сильно зашумлен. Поэтому нам нужно добавить в анализ дополнительный набор бумаг, чтобы их было порядка 20. С этой целью критерием отбора будут бумаги с дневным оборотом > 100-200 млн рублей и спредом < 10 bps. К таким можно отнести голубые фишки: SBER, GAZP, LKOH, NVTK, MGNT, ROSN, TATN, GMKN, PLZL, VTBR, MOEX, SNGS, SNGSP, SBERP, а также бумаги второго эшелона: IRAO, TRNFP, PHOR, AFLT, MAGN, BSPB. На основе этого набора загрузим данные по ценам с сайта Московской биржи.

In [9]:
import pandas as pd
import requests
import time
import os

# даты конца/начала рядов
date_start = '2015-01-01'
date_end = '2024-12-31'
# тикеры
tickers = [
    'SBER', 'GAZP', 'LKOH', 'NVTK', 'MGNT', 'ROSN', 'TATN', 'GMKN', 'PLZL',
    'VTBR', 'MOEX', 'SNGS', 'SNGSP', 'SBERP',
    # второй эшелон
    'TRNFP', 'PHOR', 'AFLT', 'MAGN', 'BSPB'
]

# функция, выполняющая загрузку конкретного тикера
def load_data(ticker, start=date_start, end=date_end):
  url = f'https://iss.moex.com/iss/engines/stock/markets/shares/securities/{ticker}/candles.json'
  all_data = []
  columns = None
  start_page = 0

  while True:
    # параметры для requests.get
    params = {
        'from': start,
        'till': end,
        'interval': 24,
        'start': start_page
    }

    try:
      r = requests.get(url, params=params, timeout=30)
      r.raise_for_status()
      j = r.json()
    except Exception as e:
      print(f'{ticker}: ошибка при загрузке на странице {start_page}: {e}')
      break

    candles = j.get('candles', {})
    if columns is None:
      columns = candles.get('columns', [])
    data = candles.get('data', [])
    if not data:
      break  # данные кончились

    all_data.extend(data) # добавляем то, что загрузили
    print(f'{ticker}: страница {start_page // 500 + 1}, всего {len(all_data)} свечей, последняя {data[-1][6]}')
    start_page += len(data)
    time.sleep(0.1)

  if not all_data:
    print(f'{ticker}: данные не загружены')
    return None

  df = pd.DataFrame(all_data, columns=columns)
  df['begin'] = pd.to_datetime(df['begin'])
  df = df.set_index('begin')
  df.index.name = 'datetime'

  return df[['open', 'high', 'low', 'close', 'volume']]  # свечи

os.makedirs('data/raw', exist_ok=True)

# загружаем тикеров
dfs = {}
for ticker in tickers:
  print(f'\n {ticker}')
  df = load_data(ticker, start=date_start, end=date_end)
  if df is not None:
    dfs[ticker] = df
    print(f'  - Загружено {df.shape} с {df.index.min().date()} по {df.index.max().date()}')

# собираем в набор: дата - тикер: данные
combined = pd.concat(dfs, axis=1)
combined.columns.names = ['ticker', 'field']
combined.index.name = 'datetime'
combined.to_parquet('data/raw/prices_moex_stable.parquet')


 SBER
SBER: страница 1, всего 500 свечей, последняя 2016-12-28 00:00:00
SBER: страница 2, всего 1000 свечей, последняя 2018-12-19 00:00:00
SBER: страница 3, всего 1500 свечей, последняя 2020-12-16 00:00:00
SBER: страница 4, всего 2000 свечей, последняя 2023-01-03 00:00:00
SBER: страница 5, всего 2500 свечей, последняя 2024-12-18 00:00:00
SBER: страница 6, всего 2509 свечей, последняя 2024-12-30 00:00:00
  - Загружено (2509, 5) с 2015-01-05 по 2024-12-30

 GAZP
GAZP: страница 1, всего 500 свечей, последняя 2016-12-28 00:00:00
GAZP: страница 2, всего 1000 свечей, последняя 2018-12-19 00:00:00
GAZP: страница 3, всего 1500 свечей, последняя 2020-12-16 00:00:00
GAZP: страница 4, всего 2000 свечей, последняя 2023-01-03 00:00:00
GAZP: страница 5, всего 2500 свечей, последняя 2024-12-18 00:00:00
GAZP: страница 6, всего 2509 свечей, последняя 2024-12-30 00:00:00
  - Загружено (2509, 5) с 2015-01-05 по 2024-12-30

 LKOH
LKOH: страница 1, всего 500 свечей, последняя 2016-12-28 00:00:00
LKOH: стр

С загрузкой сложностей не возникло. Теперь было бы неплохо визуализировать данные, с которыми мы работаем. Сделаем это при помощи программы ниже.

Можно использовать японские свечи, но на таком временном периоде масштаб будет слишком мелким. Поэтому ограничимся ценами закрытия и скользящими средними. Цены закрытия смотрим не случайно - ребалансировка портфеля осуществляется на основании данных за конец дня. Но в целом, здесь для визуализации можно было использовать и другие цены.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

def make_plot(ticker: str, close: pd.DataFrame, volume: pd.DataFrame = None,
                  start=None, end=None, maw=(20, 50), save_dir=None):
  if ticker not in close.columns:
    print(f'{ticker}: нет данных')
    return

  s = close[ticker].dropna()
  # отбираем дни
  if start is not None:
    s = s[s.index >= pd.Timestamp(start)]
  if end is not None:
    s = s[s.index <= pd.Timestamp(end)]

  if s.empty:
    print(f'{ticker}: нет данных в указанном периоде')
    return

  # в зависимости от того, хотим отображать объём или нет
  if volume is not None and ticker in volume.columns:
    fig, (ax1, ax2) = plt.subplots(
        nrows=2, ncols=1, figsize=(14, 7), sharex=True,
        gridspec_kw={'height_ratios': [3, 1]}
    )
  else:
    fig, ax1 = plt.subplots(figsize=(14, 6))
    ax2 = None

  ax1.plot(s.index, s.values, color='steelblue',
           linewidth=1.1, label=f'{ticker} - цена закрытия')

  # скользящие средние
  if maw:
    for w in maw:
      ma = s.rolling(w).mean()
      ax1.plot(ma.index, ma.values, linewidth=1, alpha=0.85, label=f'MA{w}')

  ax1.set_title(f'{ticker}: данные с {s.index.min().date()} по {s.index.max().date()}')
  ax1.set_ylabel('Цена закрытия, руб')
  ax1.grid(True, alpha=0.3)
  ax1.legend(loc='upper left', fontsize=9)

  if ax2 is not None:
    v = volume[ticker].reindex(s.index).fillna(0)
    ax2.bar(v.index, v.values, color='gray', width=1, alpha=0.7)
    ax2.set_ylabel('Объём')
    ax2.set_xlabel('Дата')
    ax2.grid(alpha=0.3)
  else:
    ax1.set_xlabel('Дата')

  last_ax = ax2 if ax2 is not None else ax1
  last_ax.xaxis.set_major_locator(mdates.YearLocator())
  last_ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

  plt.tight_layout()

  if save_dir:
      os.makedirs(save_dir, exist_ok=True)
      fname = os.path.join(save_dir, f'{ticker}_close.png')
      fig.savefig(fname, dpi=140, bbox_inches='tight')
      plt.close(fig)
      print(f'Цены закрытия для {ticker} сохранены в {fname}')
  else:
      plt.show()

path = 'data/raw/prices_moex_stable.parquet'

if not os.path.exists(path):
  raise FileNotFoundError(
      f'Не удалось загрузить файл {path}'
  )

combined = pd.read_parquet(path)
close = combined.xs('close', axis=1, level='field')
volume = combined.xs('volume', axis=1, level='field')

tickers = close.columns.tolist()
print(f'Тикеры: {tickers}')

folder = 'data/charts/close'  # для запуска в IDE и сохранения

for t in tickers:
  make_plot(t, close, volume, maw=(50,))


Из этих многочисленных графиков наиболее интересный эффект - обвал в начале 2022 года и резкий рост объёма торгов. Это, кстати, и есть прямое проявление новостного фона, которое было бы неплохо интегрировать в стратегию прогнозирования с помощью нейронных сетей. Например, можно анализировать как положительность/отрицательность новости, так и степень её влияния на рынок. В целом, простора для размышлений здесь много, но пока попробуем адаптировать анализ с целью дальнейшего обучения моделей на основе технических данных. Далее в последующих блокнотах мы реализуем ряд признаков на основе наших временных рядов, что будет иметь принципиальное значение в дальнейшем.

Среди используемых признаков будут: моментум-признаки (доходность за 1, 5, 10, 20 дней; кросс-секицонные доходности, то есть доходности каждой бумаги относительно среднего по рынку за выбранный период, нормированный на стандартное отклонение по всем бумагам - 1, 5, 20 дней), ревёрсал-признаки (разность между ценой закрытия и наименьшей ценой, делённая на разность наибольшей и наименьшей - если цена закрытия близка к наивысшей, то, как правило, на следующий день будет откат; разница между ценой закрытия и скользящим средним за 20 дней, нормированная на бегущее стандартное отклонение за 20 дней - бумаги, идущие далеко от ma20, как правило, возвращаются обратно), волатильность/риск (реализованная волатильность за 20 дней - бегущее стандартное отклонение логарифмических доходностей за 20 дней; волатильность с учётом гэпов - сглаживание Уайлдера, рассчитанное на гэпах, за 14 дней, нормированное на цену закрытия (для устранения влияния цены), - для улавливания внутридневных волатильностей;  кросс-секционный z-score волатильности - аналогично доходности, - вясняем, насколько волатильность отличается от среднего, будет нужно для расчёта IC; отношения короткой и длинной волатильностей (за 5 и 20 дней) - ловим начало периодов волатильности (кластеризация волатильностей); бета к рынку - отношение скользящей ковариации логарифмических доходностей за 20 дней с тем же показателем для рынка к скользящей дисперсии рыночной логарифмической доходности за тот же период; корреляция с рынком - скользящая ковариация логарифмических доходностей за 20 дней с аналогичным показателем для рынка, нормированный на скользящие стандартные отклонения доходностей и рынка за тот же период), осцилляторы (RSI(14) - доходности + сглаживание Уайлдера для придания большего веса последним дням -> relative strength -> RSI - перекупленность/перепроданность; кросс-секционный RSI(14)), объём (OBV MA20 - разность объёма с бегущим средним объёмом за 20 дней, нормированная на стандартное отклонение за 20 дней в объёмах; кросс-секционный volume_z, - отражают, насколько сильный сигнал). Все эти признаки реализованы в программе ниже.

In [10]:
import numpy as np

# вспомогательные функции
# вернуть нужное поле для всех тикеров
def _wide(prices, field):
  return prices.xs(field, level=1, axis=1)

# RSI со сглаживанием Уайлдера
def _wilder_rsi(close, n=14):
  delta = close.diff()
  gain = delta.clip(lower=0)
  loss = -delta.clip(upper=0)
  ag = gain.ewm(alpha=1/n, min_periods=n, adjust=False).mean()
  al = loss.ewm(alpha=1/n, min_periods=n, adjust=False).mean()
  rs = ag / al
  RSI = 100 - 100 / (1 + rs)

  return RSI

# нормированные величины
def _cs_zscore(df):
  mu = df.mean(axis=1)
  std = df.std(axis=1)
  return df.sub(mu, axis=0).div(std, axis=0)

# ранги
def _cs_rank(df):
  return df.rank(axis=1, pct=True) - 0.5

# безопасное деление
def _save_div(a, b):
  return a / b.replace(0, np.nan)

# кросс-секционные
def add_cross_sectional_features(tech_features):
  cs = {}
  skip = {'obv'}
  for name, df in tech_features.items():
    if name in skip:
      continue
    cs[f'cs_{name}_z'] = _cs_zscore(df)
    cs[f'cs_{name}_rank'] = _cs_rank(df)

  return cs


# признаки
def add_technical_features(prices):
  # извлекаем признаки
  close = _wide(prices, 'close')
  open_ = _wide(prices, 'open')
  high = _wide(prices, 'high')
  low = _wide(prices, 'low')
  volume = _wide(prices, 'volume')

  f = {}

  # доходности
  for i in [1, 5, 10, 20]:
    f[f'ret_{i}d'] = close.pct_change(i, fill_method=None)

  # логарифмические доходности
  log_ret = np.log(close / close.shift(1))
  f['log_ret_1d'] = log_ret

  # волатильности
  for i in [5, 20]:
    f[f'vol_{i}d'] = log_ret.rolling(i, min_periods=i).std()

  # отношение волатильностей
  f['vol_ratio_5_20'] = _save_div(f['vol_5d'], f['vol_20d'])

  # RSI14
  f['rsi_14'] = _wilder_rsi(close, 14)

  # ATR
  tr = pd.concat([
      high - low,
      (high - close.shift(1)).abs(),
      (low - close.shift(1)).abs()
  ]).groupby(level=0).max()
  f['atr_14'] = tr.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
  f['atr_norm'] = _save_div(f['atr_14'], close)

  # close position
  rng = high - low
  f['close_position'] = _save_div(close - low, rng)

  ma20 = close.rolling(20, min_periods=20).mean()
  std20 = close.rolling(20, min_periods=20).std()
  f['boll_pos'] = _save_div(close - ma20, 2 * std20)

  # объёмы
  vol20ma = volume.rolling(20, min_periods=20).mean()
  vol20std = volume.rolling(20, min_periods=20).std()
  f['volume_z'] = _save_div(volume - vol20ma, vol20std)

  # OBV
  sign = np.sign(close.diff())
  obv = (sign * volume).fillna(0).cumsum()
  f['obv_z'] = _save_div(obv - obv.rolling(20).mean(), obv.rolling(20).std())

  return f

# рыночные признаки
def add_market_features(prices):
  close = _wide(prices, 'close')
  log_ret = np.log(close / close.shift(1))

  mkt_ret = log_ret.mean(axis=1)

  f = {}

  cov = log_ret.rolling(20, min_periods=20).cov(mkt_ret)
  var = log_ret.rolling(20, min_periods=20).var()
  f['beta_20d'] = cov.div(var, axis=0)
  f['corr_20d'] = log_ret.rolling(20, min_periods=20).corr(mkt_ret)

  return f


# строим признаки
def add_features(prices):
  tech = add_technical_features(prices)
  cs = add_cross_sectional_features(tech)
  mkt = add_market_features(prices)

  all_feats = {**tech, **cs, **mkt}

  tickers = prices.columns.get_level_values(0).unique()
  out = {}
  for t in tickers:
    df_t = pd.DataFrame({name: f[t] for name, f in all_feats.items()})
    out[t] = df_t
  result = pd.concat(out, axis=1)
  result.columns.names = ['ticker', 'feature']

  return result


prices = pd.read_parquet('data/raw/prices_moex_stable.parquet')
features = add_features(prices)

print(features.shape)
print(features.head(25))



(2509, 893)
ticker          SBER                                                     \
feature       ret_1d    ret_5d   ret_10d   ret_20d log_ret_1d    vol_5d   
datetime                                                                  
2015-01-05       NaN       NaN       NaN       NaN        NaN       NaN   
2015-01-06  0.033883       NaN       NaN       NaN   0.033322       NaN   
2015-01-08  0.127316       NaN       NaN       NaN   0.119840       NaN   
2015-01-09 -0.039574       NaN       NaN       NaN  -0.040378       NaN   
2015-01-12 -0.003170       NaN       NaN       NaN  -0.003175       NaN   
2015-01-13 -0.041176  0.069895       NaN       NaN  -0.042048  0.067022   
2015-01-14 -0.004311  0.030371       NaN       NaN  -0.004320  0.066351   
2015-01-15  0.029142 -0.059361       NaN       NaN   0.028726  0.029595   
2015-01-16  0.003074 -0.017591       NaN       NaN   0.003070  0.025340   
2015-01-19 -0.003065 -0.017488       NaN       NaN  -0.003070  0.025340   
2015-01-20 -0

Нашей последующей задачей, как и ранее, будет формирование целевых переменных. В этом плане здесь всё относительно просто: в основном мы будем использовать доходность вперёд (forward return - сырая доходность), рассчитанная как разность ценами закрытия между днём t и t+5, нормированная на цену закрытия дня t. Чтобы получить такой ряд, мы просто сдвинем значения pct_change(5) влево на 5 шагов. Это основная целевая переменная для регрессии, так что у нас будет возможность предсказывать величину доходности. Но этого может оказаться и недостаточно, так как модель может тратить ёмкость на предсказание волатильности, а не направления. Поэтому мы добавим risk-adjusted forward return - отношение forward return за 20 дней к реализованной волатильности за тот же период (в день t, иначе утечка), - для устранения эффекта волатильности. Кроме того, мы добавляем кросс-секционный ранг доходности - отношение ранга бумаги в день t по доходности ко дню t+5 к их количеству, из которой вычтем 0.5 для симметричного диапазона. Эта целевая переменная может хорошо использоваться при формировании портфеля, а также для ражирования. Последняя переменная - классификация вверх/вниз на основе доходности.

В первом варианте нам удалось получить IC-mean 0.0675 на y_ret. Показатель хороший, но к нему есть несколько вопросов. Всё-таки, велик риск, что модель научилась предсказывать именно волатильность, а не ражировку бумаг. В подтверждение этой мысли выступает и плохие результаты при формировании портфеля, так как при формировании портфеля нам нужны ранги, а не волатильность. Это и могло стать причиной провала стратегии формирования портфеля. В то же время нам стоит пересмотреть применяемые модели, а точнее: мы использовали стандартный градиентный бустинг, который оптимизирует MSE. MSE на рангах не равен ранговой корреляции. Для оптимизации IC нам нужны другие функции потерь: LambdaMART, pairwise ranking loss, softrank. Так что пока оставим те же целевые пременные, но доработаем модели обучения.

In [11]:
def make_targets(prices, horizon=5):
  close = _wide(prices, 'close')

  log_ret = np.log(close / close.shift(1))
  vol_20d = log_ret.rolling(20, min_periods=20).std()

  # доходность на интервале 5 дней
  y_reg = close.pct_change(horizon, fill_method=None).shift(-horizon)
  # risk-adjusted forward return
  y_adj = _save_div(y_reg, vol_20d)
  # кросс-секционный ранг доходности
  y_rank = y_reg.rank(axis=1, pct=True) - 0.5

  y_cls = pd.DataFrame(index=y_reg.index, columns=y_reg.columns, dtype='Int64')
  y_cls[y_reg > 0] = 1
  y_cls[y_reg < 0] = 0

  return y_reg, y_adj, y_rank, y_cls

prices = pd.read_parquet('data/raw/prices_moex_stable.parquet')
y_reg, y_adj, y_rank, y_cls = make_targets(prices)

# проверим загрузку
print(' y_reg - регрессия')
print(y_reg.head(10))
print('\n y_cls - классификация')
print(y_cls.head(10))
# ещё раз отобразим баланс классов
print('\n', y_cls.apply(pd.Series.value_counts, normalize=True).round(3))

 y_reg - регрессия
ticker          SBER      GAZP      LKOH      NVTK      MGNT      ROSN  \
datetime                                                                 
2015-01-05  0.069895  0.064875  0.098911  0.074424  0.131923  0.041423   
2015-01-06  0.030371  0.053124  0.103625  0.044472  0.100962  0.045931   
2015-01-08 -0.059361  0.003687  0.018235  0.000414  0.063706  0.023029   
2015-01-09 -0.017591  0.055752  0.111743  0.031362  0.076646  0.093901   
2015-01-12 -0.017488  0.056269  0.065805  0.034629  0.051081  0.104412   
2015-01-13  0.008125  0.021803  0.042308 -0.029424  0.019231  0.107857   
2015-01-14  0.045795  0.017088  0.063369 -0.015063  0.035371  0.119809   
2015-01-15  0.041262  0.030612  0.076788  0.083816  0.067764  0.099591   
2015-01-16  0.039684  0.021591  0.041301  0.044898  0.035242  0.079204   
2015-01-19 -0.021197 -0.000743  0.085189  0.034908  0.027147  0.053040   

ticker          TATN      GMKN      PLZL      VTBR      MOEX      SNGS  \
datetime          

Данные выглядят правдоподобно, только в IRAO очень много NaN. Поэтому этого эмитента лучше пока удалим (видимо, он торговался не с 2015 года). Другие NaN будем удалять в дальнейшем.

Теперь вновь займёмся формированием flat-датасета. Существенных изменений здесь не будет.

In [12]:
# сборка flat-датасета
def build_flat(features, y_cls, y_reg, y_adj, y_rank, feature_cols):
  parts = []
  # формируем таблицу признаков для каждого тикера
  for t in y_cls.columns:
    f = features[t][feature_cols].copy()
    f['target_cls'] = y_cls[t]
    f['target_reg'] = y_reg[t]
    f['target_adj'] = y_adj[t]
    f['target_rank'] = y_rank[t]
    f['ticker'] = t
    parts.append(f)

    flat = pd.concat(parts, axis=0)
    # так как проще говорить о временном ряде для тикера - swaplevel
    flat = flat.set_index('ticker', append=True).swaplevel(i=0, j=1)
    flat.index = flat.index.set_names(['ticker', 'datetime'])
    flat = flat.dropna()
    flat['target_cls'] = flat['target_cls'].astype(int)

  return flat

# отберём признаки, как и ранее
feature_cols = [
    # моментум
    'ret_1d', 'ret_5d', 'ret_10d', 'ret_20d',
    'cs_ret_1d_z', 'cs_ret_5d_z', 'cs_ret_20d_z',
    # ревёрсал
    'close_position',
    'cs_boll_pos_z',
    # волатильность/риск
    'vol_20d', 'atr_norm',
    'cs_vol_20d_z',
    'vol_ratio_5_20',
    'beta_20d', 'corr_20d',
    # осцилляторы
    'rsi_14', 'cs_rsi_14_z',
    # объёмы
    'volume_z', 'cs_volume_z_z'
]

flat = build_flat(features, y_cls, y_reg, y_adj, y_rank, feature_cols)

# поделим на обучающие и тестовые наборы
train_end = pd.Timestamp('2021-12-20')  # промежуток 5 дней, как и было
test_start = pd.Timestamp('2022-01-01')

train = flat[flat.index.get_level_values('datetime') <= train_end]
test = flat[flat.index.get_level_values('datetime') >= test_start]

print(f'Train: {train.shape}')
print(f'Test: {test.shape}')

Train: (32848, 23)
Test: (13897, 23)


Здесь для удобства мы разделим ячейки с обучением и формированием flat-датасета, так как теперь самостоятельный интерес представляет обучение моделей ML. На этот раз мы реализуем сразу множество алгоритмов ML с целью выявления наиболее результативных показателей (в первую очередь - на y_rank, т.к. для формирования портфеля важна скорее эта переменная). Этим мы займёмся в следующей ячейке.

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor, ExtraTreesRegressor,
                              RandomForestClassifier, GradientBoostingClassifier)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
import xgboost as xgb
import lightgbm as lgb
import pandas as pd
import numpy as np

# регрессоры
def make_regressors():
  return {
      # линейные показали слабые результаты
      # деревья
      'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=3,
                                            min_samples_leaf=50, n_jobs=-1,
                                            random_state=42),
      'ExtraTrees': ExtraTreesRegressor(n_estimators=200, max_depth=3,
                                        min_samples_leaf=50, n_jobs=-1,
                                        random_state=42),
      'GBM': GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                       max_depth=3, subsample=0.8, min_samples_leaf=50,
                                       random_state=42),
      'HistGBM': HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05,
                                               max_depth=3, min_samples_leaf=50,
                                               random_state=42),
      # ближайшие соседи
      'KNN': KNeighborsRegressor(n_neighbors=20, weights='distance', n_jobs=-1),
      # бустинги
      'XGBoost': xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=3,
                                    subsample=0.8, colsample_bytree=0.8,
                                    reg_alpha=0.1, reg_lambda=1.0,
                                    random_state=42, n_jobs=-1, verbosity=0),
      'LightGBM': lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=3,
                                      num_leaves=15, min_child_samples=50,
                                      subsample=0.8, colsample_bytree=0.8,
                                      random_state=42, n_jobs=-1, verbose=-1)
  }

# классификаторы
def make_classifiers():
  return {
    'LogReg': LogisticRegression(max_iter=1000, random_state=42),
    'RFCls': RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=50,
        n_jobs=-1, random_state=42),
    'GBMCls': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3,
        subsample=0.8, random_state=42),
    'XGBCls': xgb.XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0),
    'LGBMCls': lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3,
        num_leaves=15, min_child_samples=50,
        random_state=42, n_jobs=-1, verbose=-1)
  }

# кросс-секционный IC за 1 день
def cross_sectional_ic(y_proba, y_reg, index, min_tickers=15):
  idx = pd.MultiIndex.from_arrays(
      [index.get_level_values('datetime'),
       index.get_level_values('ticker')],
      names=['datetime', 'ticker']
  )

  df = pd.DataFrame({'proba': np.asarray(y_proba),
                     'y_reg': np.asarray(y_reg)},
                     index=idx)

  def ic_per_day(g):
    if len(g) < min_tickers:
      return np.nan
    if g['proba'].nunique() < 2 or g['y_reg'].nunique() < 2:
      return np.nan
    return spearmanr(g['proba'], g['y_reg'])[0]

  ic_series = df.groupby(level='datetime').apply(ic_per_day).dropna()
  std = ic_series.std()
  mean = ic_series.mean()

  return {
      'ic_mean': mean,
      'ic_std': std,
      'ic_ir': mean / std,
      'ic_positive_share': (ic_series > 0).mean(),
      'n_days': len(ic_series),
      'ic_series': ic_series
  }

# расчёт метрик
def metrics(y_true, y_pred, y_proba, y_reg_true, index, name):
  auc = roc_auc_score(y_true, y_proba)

  ic_stats = cross_sectional_ic(y_proba, y_reg_true, index, min_tickers=15)
  ic_mean = ic_stats['ic_mean']
  ic_ir = ic_stats['ic_ir']
  ic_pos = ic_stats['ic_positive_share']
  n_days = ic_stats['n_days']

  print(f"\n{name}:")
  print(f"ROC-AUC:  {auc:.4f}")
  print(f"IC mean:  {ic_mean:.4f}")
  print(f"IC std:   {ic_stats['ic_std']:.4f}")
  print(f"ICIR:     {ic_ir:.4f}")
  print(f"%IC > 0:  {ic_pos:.3f}")
  print(f"n_days:   {n_days}")

  return {
        'name': name, 'auc': auc,
        'ic_mean': ic_mean, 'ic_ir': ic_ir,
        'ic_pos': ic_pos, 'n_days': n_days,
        'ic_series': ic_stats['ic_series']
  }


X_train = train[feature_cols].values
X_test = test[feature_cols].values

# стандартизированные
sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s = sc.transform(X_test)

test_index = test.index

all_results = []
all_preds = {}

y_test_reg = test['target_reg'].values

print('РЕГРЕССИЯ')

for target in ['target_reg', 'target_adj']:
    y_train = train[target].values
    print(f'\n {target}:\n')

    # теперь регрессоры
    for name, model in make_regressors().items():
        try:  # была проблема с долгой работой
            model.fit(X_train_s, y_train)
            pred = model.predict(X_test_s)
            all_preds[(name, target)] = pred

            ic = cross_sectional_ic(pred, y_test_reg, test_index)
            all_results.append({
                'model': name, 'target': target,
                'IC_mean': ic['ic_mean'], 'IC_std': ic['ic_std'],
                'ICIR': ic['ic_ir'], 'IC_pos': ic['ic_positive_share'],
                'n_days': ic['n_days']
            })
        except Exception as e:
            print(f'Возникло ислючение: {e} на {name}')
            continue

        print(f'  {name:<14} IC={ic['ic_mean']:>7.4f} ICIR={ic['ic_ir']:>7.4f} %IC>0={ic['ic_positive_share']:.3f}')

print('\nКЛАССИФИКАЦИЯ')

y_train_cls = train['target_cls'].values

for name, model in make_classifiers().items():
    try:
        model.fit(X_train_s, y_train_cls)
        proba = model.predict_proba(X_test_s)[:, 1]
        all_preds[(name, 'target_cls')] = proba

        ic = cross_sectional_ic(proba, y_test_reg, test_index)
        auc = roc_auc_score(test['target_cls'].values, proba)

        all_results.append({
            'model': name, 'target': 'target_cls',
            'IC_mean': ic['ic_mean'], 'IC_std': ic['ic_std'],
            'ICIR': ic['ic_ir'], 'IC_pos': ic['ic_positive_share'],
            'n_days': ic['n_days']
        })
    except Exception as e:
        print(f'Возникло исключение: {e} на {name}')
        continue

    print(f'  {name:<14} IC={ic['ic_mean']:>7.4f} ICIR={ic['ic_ir']:>7.4f} %IC>0={ic['ic_positive_share']:.3f} AUC={auc:.4f}')

# создадим pivot-таблицу
results_df = pd.DataFrame(all_results)
pivot = results_df.pivot_table(index='model', columns='target',
                                 values='IC_mean').sort_values('target_reg',
                                                                ascending=False,
                                                                na_position='last')
print('\nТаблица: модель * целевая переменная')
print(pivot.round(4).to_string())

# лучшие 10 комбинаций
print('\nЛучшие 10 комбинаций')
top10 = results_df.sort_values('IC_mean', ascending=False).head(10)
print(top10[['model', 'target', 'IC_mean', 'ICIR', 'IC_pos']].round(4).to_string(index=False))

# лучшие на y_reg
top3reg = results_df[results_df['target'] == 'target_reg'].sort_values('IC_mean', ascending=False).head(3)['model'].to_list()
print(f'Лучшие 3 на target_reg: {top3reg}')

preds_3reg = np.column_stack([all_preds[(m, 'target_reg')] for m in top3reg])

for agg_name, agg_fn in [('mean', np.mean), ('median', np.median)]:
  ens = agg_fn(preds_3reg, axis=1)
  ic_ens = cross_sectional_ic(ens, y_test_reg, test_index)
  print(f'Ансамбль {agg_name}: IC={ic_ens['ic_mean']:.4f} ICIR={ic_ens['ic_ir']:.4f} %IC>0={ic_ens['ic_positive_share']:.3f}')

РЕГРЕССИЯ

 target_reg:

  RandomForest   IC=-0.0132 ICIR=-0.0517 %IC>0=0.495
  ExtraTrees     IC= 0.0340 ICIR= 0.1234 %IC>0=0.556
  GBM            IC= 0.0220 ICIR= 0.0847 %IC>0=0.532
  HistGBM        IC= 0.0171 ICIR= 0.0627 %IC>0=0.524
  KNN            IC=-0.0097 ICIR=-0.0365 %IC>0=0.484
  XGBoost        IC= 0.0148 ICIR= 0.0564 %IC>0=0.535


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  LightGBM       IC= 0.0108 ICIR= 0.0398 %IC>0=0.512

 target_adj:

  RandomForest   IC=-0.0021 ICIR=-0.0080 %IC>0=0.475
  ExtraTrees     IC=-0.0127 ICIR=-0.0457 %IC>0=0.475
  GBM            IC= 0.0084 ICIR= 0.0320 %IC>0=0.502
  HistGBM        IC= 0.0141 ICIR= 0.0545 %IC>0=0.524
  KNN            IC=-0.0057 ICIR=-0.0221 %IC>0=0.487
  XGBoost        IC=-0.0000 ICIR=-0.0001 %IC>0=0.506


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  LightGBM       IC= 0.0106 ICIR= 0.0406 %IC>0=0.517

КЛАССИФИКАЦИЯ
  LogReg         IC= 0.0010 ICIR= 0.0035 %IC>0=0.499 AUC=0.5102
  RFCls          IC= 0.0135 ICIR= 0.0516 %IC>0=0.516 AUC=0.5189
  GBMCls         IC= 0.0189 ICIR= 0.0750 %IC>0=0.541 AUC=0.5221
  XGBCls         IC= 0.0213 ICIR= 0.0860 %IC>0=0.522 AUC=0.5201


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  LGBMCls        IC= 0.0267 ICIR= 0.1090 %IC>0=0.543 AUC=0.5246

Таблица: модель * целевая переменная
target        target_adj  target_cls  target_reg
model                                           
ExtraTrees       -0.0127         NaN      0.0340
GBM               0.0084         NaN      0.0220
HistGBM           0.0141         NaN      0.0171
XGBoost          -0.0000         NaN      0.0148
LightGBM          0.0106         NaN      0.0108
KNN              -0.0057         NaN     -0.0097
RandomForest     -0.0021         NaN     -0.0132
GBMCls               NaN      0.0189         NaN
LGBMCls              NaN      0.0267         NaN
LogReg               NaN      0.0010         NaN
RFCls                NaN      0.0135         NaN
XGBCls               NaN      0.0213         NaN

Лучшие 10 комбинаций
     model     target  IC_mean   ICIR  IC_pos
ExtraTrees target_reg   0.0340 0.1234  0.5562
   LGBMCls target_cls   0.0267 0.1090  0.5426
       GBM target_reg   0.0220 0.0847  0.5318
    XG

С точки зрения формирования портфеля полезнее скорее полезнее ранжировать бумаги по их потенциальному доходу. Поэтому, в целом, от y_rank всё же можно отказаться. Из результатов обучения имеет смысл далее оптимизировать модели с IC_mean > 0.03, что свидетельствует о наличии сигнала. Как мы видим, по этому критерию подходят модели ExtraTrees, GBM (у него IC < 0.03, но стоит поизучать) на target_reg, HistGBM на target_adj и target_reg. В целом, для портфельного анализа классификаторы вверх/вниз дадут мало полезной информации, но всё же ради интереса мы рассмотрели и их показатели, но далее мы исключим их из рассмотрения.

Теперь нашей дальнейшей целью будет оптимизация указанных моделей на соответствующих таргетах с помощью, например, поиска по сетке. Результаты этих манипуляций приведены ниже.